In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pprint

from generate_simple_panel import generate_panel
from make_split import SplitGenerator

In [3]:
panel = generate_panel()

In [4]:
print(panel.columns)
print()
print(panel.shape)
print()
print(panel.head())

Index(['firm_id', 't', 'y_1', 'y_2', 'tratado', 'control', 'cohort'], dtype='str')

(3086, 7)

   firm_id  t    y_1     y_2  tratado  control  cohort
0        0  0  7.367  46.123     True    False       0
1        0  1  9.411  46.683     True    False       0
2        0  2  7.898  53.131     True    False       0
3        0  3  9.999  47.314     True    False       0
4        0  4  8.462  50.540     True    False       0


In [5]:
n_treated = 0
n_control = 0
n_cohort = {}

for group, data in panel.groupby("firm_id"):
    is_treated = int(data["tratado"].iloc[0])
    is_control = int(data["control"].iloc[0])
    cohort = int(data["cohort"].iloc[0])

    n_treated += is_treated
    n_control += is_control

    if cohort >= 0:
        if cohort not in n_cohort:
            n_cohort[cohort] = {}
        n_cohort[cohort]["treated"] = n_cohort[cohort].get("treated", 0) + is_treated
        n_cohort[cohort]["control"] = n_cohort[cohort].get("control", 0) + is_control

print(f"Number of treated firms: {n_treated}")
print(f"Number of control firms: {n_control}")
print(f"Number of cohorts: {len(n_cohort)}")
print("Cohort sizes:")
pprint.pprint(n_cohort)

Number of treated firms: 60
Number of control firms: 60
Number of cohorts: 3
Cohort sizes:
{0: {'control': 20, 'treated': 20},
 1: {'control': 20, 'treated': 20},
 2: {'control': 20, 'treated': 20}}


In [9]:
split_generator = SplitGenerator(panel, train_nini_ratio=0.5, seed=42)
split = split_generator.generate()
print(split)

{'train': {'T': [0, 2, 3, 6, 8, 12, 16, 24, 25, 26, 29, 30, 33, 46, 47, 49, 50, 51, 55, 56, 61, 69, 73, 77, 82, 84, 96, 97, 101, 103, 107, 110, 111, 114, 116, 118, 127, 129, 132, 137, 139, 142, 147, 149, 160, 162, 163, 164, 166, 173, 177, 178, 181, 182, 185, 188, 189, 190, 194, 195], 'NiNi': [85, 130, 106, 76, 153, 67, 1, 151, 60, 93, 52, 109, 70, 143, 128, 11, 72, 66, 48, 18, 78, 174, 198, 184, 140, 170, 13, 4, 145, 112, 7, 120, 155, 180, 59, 161, 176, 83, 95, 63]}, 'test': {'C': [5, 10, 15, 17, 19, 22, 27, 28, 34, 35, 36, 38, 39, 40, 42, 45, 53, 57, 58, 64, 65, 68, 71, 74, 75, 80, 86, 87, 89, 90, 98, 99, 100, 102, 104, 105, 115, 117, 119, 123, 126, 131, 134, 135, 144, 146, 148, 152, 156, 165, 168, 169, 172, 175, 179, 186, 187, 191, 193, 199], 'NiNi': [192, 150, 43, 21, 81, 133, 44, 113, 124, 121, 141, 154, 9, 88, 79, 23, 167, 183, 138, 125, 14, 94, 171, 31, 197, 108, 62, 54, 91, 157, 122, 32, 136, 41, 37, 159, 92, 158, 196, 20]}, 'meta': {'train_nini_ratio': 0.5, 'seed': 42, 'n_train

In [13]:
train = split['train']
test = split['test']

treated = train['T']
nini_train = train['NiNi']

control = test['C']
nini_test = test['NiNi']

## Chequeo de seguridad
for id in treated:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["tratado"].iloc[0] == True, f"Firm {id} is not treated"

for id in control:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["control"].iloc[0] == True, f"Firm {id} is not control"

for id in nini_train + nini_test:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["tratado"].iloc[0] == False and firm["control"].iloc[0] == False, f"Firm {id} is not NiNi"